# GPT-2 Medium vs Word2Vec on EWoK BabyLM Completion Choice

This notebook compares the final GPT-2 Medium checkpoint against the final FineWeb-Edu Word2Vec run on the **BabyLM completion-choice** metric.

It focuses on three questions:
1. Are the two models correlated on a per-item basis within each domain?
2. In a lexical-friendly domain such as `material-dynamics`, where do they fail together or differ?
3. For selected items, how does GPT distribute token-level log-probability mass across `T_1` vs `T_2` under `C_1` (and symmetrically under `C_2`)?

In [ ]:
import importlib
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.w2v_lexical_probe import gpt_w2v_babylm_completion_analysis as babylm_analysis

babylm_analysis = importlib.reload(babylm_analysis)

DEFAULT_GPT_RUN_DIR = babylm_analysis.DEFAULT_GPT_RUN_DIR
DEFAULT_W2V_RUN_DIR = babylm_analysis.DEFAULT_W2V_RUN_DIR
build_domain_correlation_significance_table = babylm_analysis.build_domain_correlation_significance_table
build_metadata_group_correlation_table = babylm_analysis.build_metadata_group_correlation_table
find_domain_failure_cases = babylm_analysis.find_domain_failure_cases
load_gpt_checkpoint_bundle = babylm_analysis.load_gpt_checkpoint_bundle
plot_domain_correlation_grid = babylm_analysis.plot_domain_correlation_grid
plot_domain_failure_quadrants = babylm_analysis.plot_domain_failure_quadrants
plot_gpt_token_logprob_panels = babylm_analysis.plot_gpt_token_logprob_panels
prepare_joint_babylm_completion_dataframe = babylm_analysis.prepare_joint_babylm_completion_dataframe
select_interesting_item = babylm_analysis.select_interesting_item
select_interesting_items = babylm_analysis.select_interesting_items
summarize_domain_correlations = babylm_analysis.summarize_domain_correlations

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)

GPT_RUN_DIR = Path(DEFAULT_GPT_RUN_DIR)
WORD2VEC_RUN_DIR = Path(DEFAULT_W2V_RUN_DIR)
EWOK_VARIANT = "fast"
GPT_SCORE_REDUCTION = "mean"
WORD2VEC_TEXT_PREPROCESSING = "probe"
FILTER_AGENT_NAMES = True
ANALYSIS_DOMAIN = "material-dynamics"
PREFER_CACHED_GPT = True

GPT_RUN_DIR, WORD2VEC_RUN_DIR

## Load and join per-item BabyLM completion scores

In [ ]:
joint_df = prepare_joint_babylm_completion_dataframe(
    gpt_run_dir=GPT_RUN_DIR,
    word2vec_run_dir=WORD2VEC_RUN_DIR,
    ewok_variant=EWOK_VARIANT,
    score_reduction=GPT_SCORE_REDUCTION,
    ewok_text_preprocessing=WORD2VEC_TEXT_PREPROCESSING,
    filter_agent_names=FILTER_AGENT_NAMES,
    prefer_cached_gpt=PREFER_CACHED_GPT,
)
joint_df.shape

## Domain-level correlation summary

This uses the **combined BabyLM completion margin** for each model:
- GPT: `0.5 * (m_1 + m_2)` from log-prob scores
- Word2Vec: `0.5 * (m_1 + m_2)` from cosine scores

In [ ]:
domain_summary = summarize_domain_correlations(joint_df)
display(domain_summary.sort_values("spearman_rho", ascending=False).reset_index(drop=True))

## Correlation significance table

This table lists each domain's correlation values, p-values, and boolean significance flags at `0.05`, `0.01`, and `0.005`.

In [ ]:
domain_significance_table = build_domain_correlation_significance_table(domain_summary)
display(domain_significance_table)

## Correlation by EWoK metadata groups

This groups items by metadata such as `ContextType`, `ContextDiff`, `TargetDiff`, and a few pairwise combinations, then recomputes the GPT-vs-Word2Vec correlation within each bucket.

The outputs are split into separate subtables so each grouping is easier to read.

In [ ]:
metadata_correlation_table = build_metadata_group_correlation_table(joint_df)

for group_name in metadata_correlation_table["group_by"].drop_duplicates().tolist():
    display(Markdown(f"### {group_name}"))
    subtable = metadata_correlation_table[metadata_correlation_table["group_by"] == group_name].copy()
    subtable = subtable.sort_values(["spearman_rho", "pearson_r"], ascending=[False, False], na_position="last")
    display(subtable.drop(columns=["group_by"]).reset_index(drop=True))

## Per-domain correlation plots

In [ ]:
fig, _axes = plot_domain_correlation_grid(joint_df)
fig

## Per-domain correlation plots with shared axes

This version keeps the same x/y limits across domains, which makes cross-domain visual comparison easier.

In [ ]:
fig, _axes = plot_domain_correlation_grid(joint_df, shared_limits=True)
fig

## Extra plot 1: failure quadrants in a lexical-friendly domain

This shows where GPT and Word2Vec both win, both miss, or disagree on `material-dynamics`.

In [ ]:
fig, _ax = plot_domain_failure_quadrants(joint_df, ANALYSIS_DOMAIN)
fig

## Failure quadrants across domains, ranked by Spearman rho

This lets us scan agreement and disagreement structure in the same order as the rank correlation results.

In [ ]:
ranked_domains = domain_summary.sort_values("spearman_rho", ascending=False)[["domain", "spearman_rho", "spearman_p"]].reset_index(drop=True)
display(ranked_domains)

for row in ranked_domains.itertuples(index=False):
    display(Markdown(f"### {row.domain}  |  Spearman rho={row.spearman_rho:.3f}, p={row.spearman_p:.3g}"))
    fig, _ax = plot_domain_failure_quadrants(joint_df, row.domain)
    display(fig)

## Inspect Word2Vec non-win cases in the chosen domain

In [ ]:
failure_cases = find_domain_failure_cases(joint_df, ANALYSIS_DOMAIN)
display(failure_cases.head(20))

## Load GPT-2 Medium for token-level inspection

This is only needed for the token log-probability plots, so it is intentionally separate from the earlier correlation analysis.

## Surface a few interesting `material-dynamics` cases

This gives us a small ranked shortlist before we drill into one item at token level.

In [ ]:
interesting_items = select_interesting_items(
    joint_df,
    domain=ANALYSIS_DOMAIN,
    strategy="word2vec_nonwin_then_gpt_nonwin",
    top_k=5,
)
display(
    interesting_items[
        [
            "row_index",
            "domain",
            "Context1",
            "Target1",
            "Context2",
            "Target2",
            "word2vec_margin_combined",
            "gpt_margin_combined",
            "word2vec_completion_sign_combined",
            "gpt_completion_sign_combined",
        ]
    ]
)

In [ ]:
gpt_bundle = load_gpt_checkpoint_bundle(GPT_RUN_DIR)
gpt_bundle.checkpoint_dir

## Select an interesting item for token-level GPT diagnostics

Default strategy: choose a `material-dynamics` item where Word2Vec is a non-win, and prefer cases where GPT is also a non-win.

In [ ]:
selected_item = interesting_items.iloc[0]
display(
    selected_item[
        [
            "row_index",
            "domain",
            "Context1",
            "Target1",
            "Context2",
            "Target2",
            "word2vec_margin_combined",
            "gpt_margin_combined",
            "word2vec_completion_sign_combined",
            "gpt_completion_sign_combined",
        ]
    ]
)

## GPT token-level completion evidence

The two panels make the BabyLM completion comparisons explicit:
- top: `T_1` and `T_2` under `C_1`
- bottom: `T_2` and `T_1` under `C_2`

In [ ]:
fig, _axes = plot_gpt_token_logprob_panels(
    gpt_bundle.model,
    gpt_bundle.tokenizer,
    selected_item,
)
fig